# Project A - main comparison

Runs compaction, the sleep consolidation loop for every method, and all evaluation arms.
Resumable: re-running picks up from the last checkpointed sleep phase.

**Run `a0_precondition.ipynb` first.** If no compactor beat the positional control there,
ours and uniform replay sample from the same distribution and this comparison is dead by
construction.

Compaction runs under Qwen2.5-1.5B (the only configuration with signal); consolidation
targets Qwen2.5-0.5B. The event log is plain text, so the two stages decouple cleanly.
Budget ~4 GPU-hours per session.

Set `DATASET` in the next cell. `hotpotqa` is the primary set — natural prose, no marker
phrase, positional control at chance. `synthetic` is for the compaction-ratio sweep and
for debugging.

In [ ]:
import glob, os, subprocess, sys, zipfile

GIT_URL = 'https://github.com/rajul-kk/context-to-weights.git'   # cleared only if you prefer a Kaggle Dataset
REPO = '/kaggle/working/myrios'
MARKER = 'baselines/cascading.py'

def looks_like_source(d):
    return os.path.exists(os.path.join(d, MARKER))

if not looks_like_source(REPO):
    src = None
    for d in sorted(glob.glob('/kaggle/input/*')):
        if looks_like_source(d):
            src = d
            break
        for z in sorted(glob.glob(os.path.join(d, '*.zip'))):
            os.makedirs(REPO, exist_ok=True)
            zipfile.ZipFile(z).extractall(REPO)
            if looks_like_source(REPO):
                src = REPO
                break
        if src:
            break
    if src and src != REPO:
        subprocess.run(['cp', '-r', src, REPO], check=True)
    if not looks_like_source(REPO) and GIT_URL:
        r = subprocess.run(['git', 'clone', GIT_URL, REPO], capture_output=True, text=True)
        print(r.stdout, r.stderr)
    assert looks_like_source(REPO), (
        'No source found. Either (a) run scripts/package_source.py locally, upload the zip '
        'as a Kaggle Dataset, and attach it via Add Input, or (b) set GIT_URL above. '
        f'Searched /kaggle/input/*, saw: {sorted(glob.glob("/kaggle/input/*"))}')

os.chdir(REPO)
sys.path.insert(0, REPO)
subprocess.run([sys.executable, '-m', 'pip', '-q', 'install', 'peft', 'accelerate', 'datasets'])

# peft raises rather than degrading when it finds an old torchao, and Kaggle ships
# 0.10.0 against a >0.16 requirement. Nothing here uses torchao, so remove it.
try:
    import importlib.metadata as _md
    _v = _md.version('torchao')
    if tuple(int(x) for x in _v.split('.')[:2]) < (0, 16):
        subprocess.run([sys.executable, '-m', 'pip', '-q', 'uninstall', '-y', 'torchao'])
        print(f'removed incompatible torchao {_v}')
except Exception:
    pass
exec(open('notebooks/_runner.py').read())
print('cwd', os.getcwd())
print('files', sorted(os.listdir('.'))[:10])
import torch
print('torch', torch.__version__, 'cuda', torch.cuda.is_available())
if not torch.cuda.is_available():
    print()
    print('=' * 68)
    print('NO GPU. Kaggle installed the CPU build of torch, so this session')
    print('has no accelerator attached. Everything below will be far too slow.')
    print()
    print('Fix: right panel -> Session options -> Accelerator -> GPU T4 x2,')
    print('then Run All again. The image swaps to a CUDA torch build on restart.')
    print('=' * 68)
else:
    print('gpu', torch.cuda.get_device_name(0),
          f'{torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')



In [ ]:
# re-run this cell (only) if the kernel restarts; it re-derives everything
import os, sys
REPO = '/kaggle/working/myrios'
os.chdir(REPO); sys.path.insert(0, REPO)
if 'run' not in dir():
    exec(open('notebooks/_runner.py').read())

RUN_ABSTAIN_ARM = True   # compaction mask as an abstention signal, see below
RUN_ORACLE_ARM  = True   # positive control: perfect selection + augmentation
N_FORMS = 20             # surface forms per fact in the oracle control
DATASET = 'synthetic'   # 'hotpotqa' | 'synthetic' | 'unmarked'

_CFG = {'hotpotqa': 'configs/kaggle_hotpotqa.yaml',
        'synthetic': 'configs/kaggle_synthetic.yaml',
        'unmarked': 'configs/kaggle_unmarked.yaml'}
_RUNS = {'hotpotqa': '/kaggle/working/artifacts/runs_hotpot',
         'synthetic': '/kaggle/working/artifacts/runs_synth',
         'unmarked': '/kaggle/working/artifacts/runs_unmarked'}
HOTPOT = DATASET == 'hotpotqa'
CFG = _CFG[DATASET]
RUNS = _RUNS[DATASET]
CASC = f'{RUNS}/cascading'
REPORT = f'{RUNS}/report'

from common.io import load_config
_c = load_config(CFG)
assert _c['run_root'] == RUNS, f"run_root mismatch: {_c['run_root']} != {RUNS}"
DATA = _c['data']['dir']
N_TRAIN, N_EVAL = 32, 48

# HotpotQA: Qwen2.5-0.5B is the better compactor (+4.77 vs +3.75 sigma) and 2.4x faster.
# synthetic/unmarked: 1.5B is the only model with signal there.
COMPACTOR = 'Qwen/Qwen2.5-0.5B-Instruct' if HOTPOT else 'Qwen/Qwen2.5-1.5B-Instruct'
TARGET = 'Qwen/Qwen2.5-0.5B-Instruct'
print(f'{DATASET}: compactor {COMPACTOR.split("/")[-1]}, target {TARGET.split("/")[-1]}, runs {RUNS}')


## Restore previous session

Attach the Kaggle Dataset holding `runs.zip` from the last session, then restore. Skips
cleanly on a first run.

In [ ]:
# Session persistence: /kaggle/working is wiped when the session restarts.
# To resume across sessions, download the runs.zip this notebook writes at the
# end (right panel -> Output), add it as an input dataset, and set ARCHIVE to it.
ARCHIVE = '/kaggle/input/CHANGE-ME/runs.zip'
import os
if os.path.exists(ARCHIVE):
    run(f"python scripts/kaggle_sync.py restore --archive {ARCHIVE} --run-root {RUNS}")
else:
    print(f'no prior archive at {ARCHIVE} - starting fresh (fine for a first run)')


In [ ]:
if HOTPOT:
    run(f"python data/load_hotpotqa.py --n-train {N_TRAIN} --n-eval {N_EVAL} --per-trajectory 4 --early-frac 1.0")
elif DATASET == 'unmarked':
    run(f"python data/generate_synthetic.py --n-train 48 --n-eval {N_EVAL} --n-turns 120 --unmarked --out artifacts/data/synthetic_hard")
else:
    run(f"python data/generate_synthetic.py --n-train 48 --n-eval {N_EVAL} --n-turns 120")


In [ ]:
run(f"python scripts/preflight.py --config {CFG}")


## Compaction - the free labels

Check the span report before continuing. Zero fallbacks, zero empty keeps, lift above the
positional control.

In [ ]:
run(f"python baselines/cascading.py --config {CFG} --split both --out {CASC} --set model.base={COMPACTOR}")
# report on eval_events - those are the trajectories the adapter consolidates
run(f"python eval/span_report.py --events {CASC}/eval_events.jsonl --show 6 --out {CASC}/eval_span_report.json")


In [ ]:
run(f"python baselines/full_context.py --config {CFG} --split eval --mode full")
run(f"python baselines/full_context.py --config {CFG} --split eval --mode none")
# reflections are built from the EVAL trajectories, because those are the
# conversations the adapter consolidates and is then tested on.
run(f"python baselines/reflection.py --config {CFG} --events {CASC}/eval_events.jsonl --out {CASC}/eval_reflections.jsonl --set model.base={COMPACTOR}")


## Sleep consolidation

`--resume` restores the adapter, the event cursor and the reservoir buffer, so an
interrupted session continues rather than restarting. Run ours and uniform first: if the
gap between them is invisible, the remaining arms are decoration.

In [ ]:
# The adapter consolidates the EVAL trajectories' compaction events and is then
# tested on those same trajectories' probes after their early facts were compacted
# away. This is within-conversation retention, per the Beyond Inference-Only setup.
# The train split is only a training-progress readout (--val-events).
def sleep_run(method, tag, extra=''):
    run_dir = f'{RUNS}/sleep_{tag}'
    refl = f'--reflections {CASC}/eval_reflections.jsonl' if method == 'reflection' else ''
    run(f"python sleep/loop.py --config {CFG} --method {method} --events {CASC}/eval_events.jsonl --val-events {CASC}/train_events.jsonl --run-dir {run_dir} --resume --val-limit 64 {refl} {extra} --set model.base={TARGET}")
    run(f"python scripts/kaggle_sync.py save --run-root {RUNS} --archive /kaggle/working/runs.zip --prune")
    return run_dir

# ours vs uniform first: if that gap is invisible the remaining arms are decoration.
sleep_run('compaction', 'compaction')
sleep_run('uniform', 'uniform')


In [ ]:
sleep_run('reflection', 'reflection')
sleep_run('compaction', 'compaction+mask', '--mask-head')

## Evaluation

Every adapter arm sees the same post-compaction context, so the adapter is the only
difference between them.

In [ ]:
assert 'RUNS' in dir(), 'run the config cell above first (kernel restarted?)'
import os

def adapter_for(tag):
    best = f'{RUNS}/sleep_{tag}/best/adapter'
    latest = f'{RUNS}/sleep_{tag}/latest/adapter'
    return best if os.path.exists(best) else latest

ARMS = [
    ('floor', f'{RUNS}/none_context/eval_contexts.jsonl', None),
    ('cascading', f'{CASC}/eval_contexts.jsonl', None),
    ('full', f'{RUNS}/full_context/eval_contexts.jsonl', None),
    ('ours', f'{CASC}/eval_contexts.jsonl', adapter_for('compaction')),
    ('uniform', f'{CASC}/eval_contexts.jsonl', adapter_for('uniform')),
    ('reflection', f'{CASC}/eval_contexts.jsonl', adapter_for('reflection')),
    ('ours+mask', f'{CASC}/eval_contexts.jsonl', adapter_for('compaction+mask')),
]
for label, ctx, adapter in ARMS:
    if not os.path.exists(ctx):
        print(f'skip {label}: no {ctx}'); continue
    if adapter and not os.path.exists(adapter):
        print(f'skip {label}: no adapter at {adapter}'); continue
    a = f'--adapter {adapter}' if adapter else ''
    run(f"python eval/retention.py --config {CFG} --contexts {ctx} --label {label} --out {REPORT}/{label} {a} --set model.base={TARGET}")


## Scoring eval (CE ranking)

Generation + string match floors at ~0 for evicted facts even when the answer is more
probable than before. This ranks the gold answer against its distractors by cross-entropy
and marks a hit when gold wins. It measures whether the knowledge is *in the weights*,
separately from whether the greedy decoder can *say* it. Synthetic only (needs known
distractor sets).

In [ ]:
if not HOTPOT:
    for label, tag in [('cascading', None), ('ours', 'compaction'), ('uniform', 'uniform'),
                       ('reflection', 'reflection'), ('ours+mask', 'compaction+mask')]:
        adp = f'--adapter {adapter_for(tag)}' if tag else ''
        run(f"python eval/scoring_retention.py --config {CFG} "
            f"--contexts {CASC}/eval_contexts.jsonl --label {label} "
            f"--out {REPORT}/score_{label} {adp} --set model.base={TARGET}")
    import glob, json
    print()
    print(f"{'arm':<12}{'MC acc':>9}{'evicted':>9}{'margin(ev)':>12}{'sigma>chance':>14}")
    for f in sorted(glob.glob(f'{REPORT}/score_*.summary.json')):
        s = json.load(open(f))
        print(f"{s['label']:<12}{s['mc_accuracy']:9.3f}{s.get('mc_accuracy_evicted', float('nan')):9.3f}"
              f"{s.get('mean_margin_evicted', float('nan')):12.3f}{s.get('evicted_sigma_over_chance', float('nan')):14.2f}")
else:
    print('scoring eval is synthetic-only (needs the generator distractor sets)')


## Abstention arm

The same free signal, the other direction. Compaction-Aware Abstention (arXiv:2608.29934)
trains a LoRA on compressor survival masks to make a 7B model *refuse* when the evidence was
evicted, and reports a 97% cut in hallucination. Consolidation asks the same mask to *recover*
the evicted fact, which the retention and scoring evals above both say it cannot.

This runs both sides on one dataset at one scale. The training label is free and identical in
each case - did this fact survive compaction - so the only thing that changes is what the
adapter is asked to do with it.

Trained on the train split, evaluated on eval. The bidirectional check matters: a model that
refuses everything scores perfectly on evicted probes, so `abstain_rate_retained` and
`accuracy_retained` have to stay healthy for the result to mean anything.


In [ ]:
if RUN_ABSTAIN_ARM:
    ABST = f'{RUNS}/abstain'
    run(f"python sleep/abstain.py --config {CFG} --contexts {CASC}/train_contexts.jsonl "
        f"--val-contexts {CASC}/eval_contexts.jsonl --run-dir {ABST} "
        f"--set model.base={TARGET}")
    for label, adp in [('no-adapter', None), ('abstain-lora', f'{ABST}/adapter')]:
        a = f'--adapter {adp}' if adp else ''
        run(f"python eval/abstention.py --config {CFG} --contexts {CASC}/eval_contexts.jsonl "
            f"--label {label} --out {REPORT}/abstain_{label} {a} --set model.base={TARGET}")
    import glob, json
    print()
    print(f"{'arm':<14}{'abstain(ev)':>12}{'abstain(ret)':>13}{'acc(ret)':>10}{'halluc(ev)':>12}{'margin':>9}")
    for f in sorted(glob.glob(f'{REPORT}/abstain_*.summary.json')):
        s = json.load(open(f))
        print(f"{s['label']:<14}{s['abstain_rate_evicted']:12.3f}{s['abstain_rate_retained']:13.3f}"
              f"{s['accuracy_retained']:10.3f}{s['hallucination_rate_evicted']:12.3f}"
              f"{s['abstention_margin']:9.3f}")
else:
    print('abstention arm skipped. Set RUN_ABSTAIN_ARM = True in the config cell.')


## Oracle control

The positive control the earlier runs lacked. Every consolidation arm scored ~0 on evicted
probes, which is uninterpretable without evidence that the setup could register a win at all.

This arm gets perfect selection (the ground-truth facts, no compactor) and Physics-of-Language-
Models-style augmentation: 20 diverse surface forms per fact, generated from templates rather
than a model, so no generator quality risk. The eval phrasing is held out - training asks
"what is the db engine?" while the eval asks "which datastore did we settle on for
notifications?" - so this tests extraction, not memorisation of the test item.

**Read it as a gate.** If the oracle beats the no-adapter baseline on evicted probes, the
benchmark can detect consolidation and the compaction-vs-uniform comparison is meaningful. If
the oracle also scores ~0, nothing at this scale can, and Project A's null says nothing about
the compaction signal.

Evaluated both closed-book and with the compacted context, since it is trained closed-book.


In [ ]:
if RUN_ORACLE_ARM and not HOTPOT:
    ORC = f'{RUNS}/oracle'
    run(f"python sleep/oracle.py --config {CFG} --trajectories {DATA}/eval.jsonl "
        f"--run-dir {ORC} --n-forms {N_FORMS} --set model.base={TARGET}")
    for label, ctx in [('oracle-closedbook', f'{RUNS}/none_context/eval_contexts.jsonl'),
                       ('oracle', f'{CASC}/eval_contexts.jsonl')]:
        if not os.path.exists(ctx):
            print(f'skip {label}: no {ctx}'); continue
        run(f"python eval/retention.py --config {CFG} --contexts {ctx} --label {label} "
            f"--out {REPORT}/{label} --adapter {ORC}/adapter --set model.base={TARGET}")
        run(f"python eval/scoring_retention.py --config {CFG} --contexts {ctx} --label {label} "
            f"--out {REPORT}/score_{label} --adapter {ORC}/adapter --set model.base={TARGET}")
    import glob, json
    print()
    print(f"{'arm':<20}{'acc':>8}{'evicted':>9}{'MC ev':>8}")
    for f in sorted(glob.glob(f'{REPORT}/*.summary.json')):
        s = json.load(open(f))
        if 'retention_accuracy' in s:
            print(f"{s['label']:<20}{s['retention_accuracy']:8.3f}"
                  f"{s.get('evicted_accuracy', float('nan')):9.3f}"
                  f"{'':>8}")
else:
    print('oracle arm skipped. Set RUN_ORACLE_ARM = True (synthetic datasets only).')


In [ ]:
run(f"python eval/significance.py --events {CASC}/eval_events.jsonl --labels compaction --out {RUNS}/significance.json")


In [ ]:
run(f"python eval/report.py --report-dir {REPORT} --run-root {RUNS} --out {RUNS}/results.md")
import os
from IPython.display import Image, Markdown, display
for fig in ['headline_retention.png', 'ce_curves.png']:
    fp = f'{REPORT}/figures/{fig}'
    if os.path.exists(fp):
        display(Image(fp))
if os.path.exists(f'{RUNS}/results.md'):
    display(Markdown(open(f'{RUNS}/results.md').read()))


In [ ]:
run(f"python scripts/kaggle_sync.py save --run-root {RUNS} --archive /kaggle/working/runs.zip --prune")
print()
print('=' * 64)
print('DOWNLOAD /kaggle/working/runs.zip NOW (right panel -> Output) or this')
print('run cannot be resumed. Then add it as a dataset and set ARCHIVE.')
print('=' * 64)
